In [1]:
# CONFIGURATION
# Modifie le symbole ici pour analyser une autre paire
# Exemples de paires majeures :
#   EURUSD   EUR/USD
#   USDJPY   USD/JPY
#   GBPUSD   GBP/USD
#   USDCHF   USD/CHF
#   AUDUSD   AUD/USD
#   USDCAD   USD/CAD
#   NZDUSD   NZD/USD
SYMBOL = "GBPUSD"
FULL_NAME = "GBP/USD"

# Timeframes
TF_DAILY  = {"interval": "1d", "period": "1mo"}
TF_H1     = {"interval": "1h", "period": "1mo"}
TF_INTRADAY = {"interval": "5m", "period": "1d"}

In [2]:
import pandas as pd
import numpy as np
import ta
from datetime import datetime

from lib.data import get_cached_data
from lib.analysis import (
    detect_candlestick_patterns,
    find_support_resistance,
    analyze_market_structure,
    build_price_action_signal,
)
from lib.ai import analyze_with_ai

In [3]:
# Donnees higher timeframe
print("=" * 60)
print(f"DAILY TIMEFRAME - {{SYMBOL}}")
print("=" * 60)
df_daily = get_cached_data(SYMBOL + "=X", TF_DAILY["interval"], TF_DAILY["period"])

print("\n" + "=" * 60)
print(f"1H TIMEFRAME - {{SYMBOL}}")
print("=" * 60)
df_1h = get_cached_data(SYMBOL + "=X", TF_H1["interval"], TF_H1["period"])

DAILY TIMEFRAME - {SYMBOL}
✓ Using cached data (age: 1m 56s)

1H TIMEFRAME - {SYMBOL}
✓ Using cached data (age: 1m 56s)


In [4]:
# Analyse daily
daily_levels = find_support_resistance(df_daily)
daily_structure = analyze_market_structure(df_daily)

# Indicateurs techniques daily
df_daily["rsi"] = ta.momentum.RSIIndicator(close=df_daily["Close"], window=14).rsi()
macd = ta.trend.MACD(close=df_daily["Close"])
df_daily["macd"] = macd.macd()
df_daily["macd_signal"] = macd.macd_signal()
df_daily["ma20"] = df_daily["Close"].rolling(20).mean()

last_d = df_daily.iloc[-1]
daily_rsi = round(float(last_d["rsi"]), 1)
daily_macd = "bullish" if last_d["macd"] > last_d["macd_signal"] else "bearish"
daily_trend = "uptrend" if float(last_d["Close"]) > float(last_d["ma20"]) else "downtrend"

print(f"RSI:      {daily_rsi}")
print(f"MACD:     {daily_macd.upper()}")
print(f"Trend:    {daily_trend.upper()}")
print()

print("Structure de marche:")
print(f"   Trend: {daily_structure['structure']}")
print(f"   Bias: {daily_structure['bias']}")
print(f"   Range: {daily_structure['price_range_pct']:.2f}%")
print()

print("Niveaux cles daily:")
for lvl in daily_levels:
    emoji = 'SUP' if lvl['type'] == 'Support' else 'RES'
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

# Analyse 1h
levels_1h = find_support_resistance(df_1h)
structure_1h = analyze_market_structure(df_1h)

print("\n" + "-" * 50)
print("1H - Structure de marche:")
print(f"   Trend: {structure_1h['structure']}")
print(f"   Bias: {structure_1h['bias']}")
print(f"   Range: {structure_1h['price_range_pct']:.2f}%")
print()

print("Niveaux cles 1H:")
for lvl in levels_1h:
    emoji = 'SUP' if lvl['type'] == 'Support' else 'RES'
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

RSI:      55.6
MACD:     BEARISH
Trend:    UPTREND

Structure de marche:
   Trend: Consolidation (Ranging)
   Bias: Neutral
   Range: 2.02%

Niveaux cles daily:
   RES Resistance: 1.35446 (Weak, 1 touches)
   SUP Support: 1.32732 (Weak, 1 touches)
   SUP Support: 1.33431 (Weak, 1 touches)

--------------------------------------------------
1H - Structure de marche:
   Trend: Consolidation (Ranging)
   Bias: Neutral
   Range: 0.58%

Niveaux cles 1H:
   RES Resistance: 1.33935 (Medium, 2 touches)
   RES Resistance: 1.34100 (Medium, 2 touches)
   RES Resistance: 1.34539 (Medium, 2 touches)
   RES Resistance: 1.34783 (Medium, 2 touches)
   SUP Support: 1.32828 (Medium, 2 touches)


In [5]:
# Donnees intraday
print("INTRADAY (5m)")
print("=" * 60)
df_5m = get_cached_data(SYMBOL + "=X", TF_INTRADAY["interval"], TF_INTRADAY["period"])

# Patterns, niveaux et structure
patterns_5m = detect_candlestick_patterns(df_5m)
levels_5m = find_support_resistance(df_5m)
structure_5m = analyze_market_structure(df_5m)

# Signal de base
signal = build_price_action_signal(df_5m, patterns_5m, levels_5m, structure_5m)

recent_patterns = [p for p in patterns_5m if p['index'] >= len(df_5m) - 5]
if recent_patterns:
    for p in recent_patterns:
        print(f"Pattern:  {p['pattern']} ({p['signal']}) - {p['strength']}")
else:
    print("Aucun pattern candlestick recent")
print()

INTRADAY (5m)
✓ Using cached data (age: 1m 56s)
Pattern:  Doji (Indecision) - Medium



In [6]:
# Enrichir le signal avec le contexte HTF
signal['pair'] = f'{SYMBOL} ({FULL_NAME})'
signal['daily_rsi'] = daily_rsi
signal['daily_macd'] = daily_macd
signal['daily_trend'] = daily_trend
signal['htf_daily_bias'] = daily_structure['bias']
signal['htf_daily_structure'] = daily_structure['structure']
signal['htf_1h_bias'] = structure_1h['bias']
signal['htf_1h_structure'] = structure_1h['structure']
signal['daily_support'] = [l['level'] for l in daily_levels if l['type'] == 'Support'][:2]
signal['daily_resistance'] = [l['level'] for l in daily_levels if l['type'] == 'Resistance'][:2]
signal['hourly_support'] = [l['level'] for l in levels_1h if l['type'] == 'Support'][:2]
signal['hourly_resistance'] = [l['level'] for l in levels_1h if l['type'] == 'Resistance'][:2]

print("\nSIGNAL COMBINE (Multi-Timeframe)")
print("=" * 70)
print(f"Daily bias:    {daily_structure['bias']:>8}  |  1H bias:      {structure_1h['bias']:>8}")
print(f"Intraday bias: {structure_5m['bias']:>8}  |  Prix: {signal['price']:.5f}")

# Verification concordance timeframes
biases = [daily_structure['bias'], structure_1h['bias'], structure_5m['bias']]
if all(b == 'Bullish' for b in biases):
    print("\nTOUS LES TIMEFRAMES SONT BULLISH - Signal haussier fort")
elif all(b == 'Bearish' for b in biases):
    print("\nTOUS LES TIMEFRAMES SONT BEARISH - Signal baissier fort")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bullish' and structure_5m['bias'] != 'Bullish':
    print("\nHTF bullish mais intraday en contretendance - Attendre confirmation")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bearish' and structure_5m['bias'] != 'Bearish':
    print("\nHTF bearish mais intraday en contretendance - Attendre confirmation")
else:
    print("\nTimeframes en desaccord - Prudence recommandee")


SIGNAL COMBINE (Multi-Timeframe)
Daily bias:     Neutral  |  1H bias:       Neutral
Intraday bias:  Neutral  |  Prix: 1.34347

Timeframes en desaccord - Prudence recommandee


In [7]:
# Analyse IA multi-timeframe

# Etape intermediaire : afficher ce qui est envoye a l'IA
print("\nPARAMETRES ENVOYES A L'IA :")
print("-" * 70)
for key, value in signal.items():
    print(f"  {key:<24} : {value}")
print("-" * 70)



PARAMETRES ENVOYES A L'IA :
----------------------------------------------------------------------
  pair                     : GBPUSD (GBP/USD)
  price                    : 1.3434721231460571
  session                  : New York
  market_structure         : Consolidation (Ranging)
  bias                     : Neutral
  price_range_pct          : 0.22
  recent_patterns          : ['Doji (Indecision)']
  nearest_support          : None
  support_strength         : None
  dist_to_support_pct      : None
  nearest_resistance       : 1.34662
  resistance_strength      : Medium
  dist_to_resistance_pct   : 0.23
  daily_rsi                : 55.6
  daily_macd               : bearish
  daily_trend              : uptrend
  htf_daily_bias           : Neutral
  htf_daily_structure      : Consolidation (Ranging)
  htf_1h_bias              : Neutral
  htf_1h_structure         : Consolidation (Ranging)
  daily_support            : [np.float64(1.32732), np.float64(1.33431)]
  daily_resistance      

In [8]:
ai_result = analyze_with_ai(signal)

if ai_result:
    emojis = {'BUY': '[BUY]', 'SELL': '[SELL]', 'HOLD': '[HOLD]'}
    ai_signal = ai_result.get('signal', 'UNKNOWN').upper()
    conf = ai_result.get('confidence', 0)

    print(f"\nSignal:       {ai_signal} {emojis.get(ai_signal, '')}")
    print(f"Confiance:    {conf}%")
    print(f"Raison:       {ai_result.get('reason', 'N/A')}")

    entry = ai_result.get('entry')
    sl = ai_result.get('stop_loss')
    tp = ai_result.get('take_profit')
    if entry is not None and sl is not None and tp is not None:
        try:
            entry, sl, tp = float(entry), float(sl), float(tp)
            print(f"Entree:       {entry:.5f}")
            print(f"Stop Loss:    {sl:.5f}")
            print(f"Take Profit:  {tp:.5f}")
        except (ValueError, TypeError):
            print(f"Entree:       {entry}")
            print(f"Stop Loss:    {sl}")
            print(f"Take Profit:  {tp}")
else:
    print("Analyse IA indisponible")



Signal:       HOLD [HOLD]
Confiance:    65%
Raison:       The market is in strong consolidation (ranging) across multiple timeframes (hourly and daily). While the daily trend is an uptrend, the current price action shows indecision (Doji), and key support/resistance levels are too close or undefined for a high-conviction directional trade. Waiting for a breakout confirmation is recommended.
